<a href="https://colab.research.google.com/github/EpicNaje23/EpicNaje23/blob/main/vehicleClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from datasets import load_from_disk
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
from google.colab import drive
from sklearn.metrics import confusion_matrix, classification_report



In [ ]:
#Monto il drive
temp = True
if  temp:
  drive.mount('/content/drive')

  # Estraiamo lo zip nello storage locale di Colab
  !unzip -q "/content/drive/MyDrive/AudioFingerprinting/dataset_raw.zip" -d /content/

Mounted at /content/drive


In [ ]:
# ============================================================
# CONFIGURAZIONE GENERALE DEL PROGETTO
# ============================================================

# Cartella principale del progetto su Drive
PROJECT_DIR = Path("/content/drive/MyDrive/AudioFingerprinting")

# Cartella RAW estratta localmente in Colab
RAW_DIR = Path("/content/dataset_raw")

# File CSV salvati su Drive
METADATA_CSV = PROJECT_DIR / "Metadata/metadata_original_files.csv"
METADATA_SPLIT_CSV = PROJECT_DIR / "Metadata/metadata_original_files_split.csv"

# Path specifici dei tre dataset
KAGGLE_DIR = RAW_DIR / "kaggle"
MELAUDIS_DIR = RAW_DIR / "melaudis"
HUGGINGFACE_DIR = RAW_DIR / "huggingface" / "test"

# Estensione audio usata nei dataset locali
AUDIO_EXTENSION = ".wav"

# Classi valide del problema finale
VALID_LABELS = {"Car", "Bus", "MC", "Tram", "Truck"}

# Mapping etichette HuggingFace
HF_LABEL_MAP = {
    0: "Car",
    1: None,
    2: "MC",
}

In [ ]:
# ============================================================
# FUNZIONI DI SUPPORTO PER I METADATI
# ============================================================

def get_audio_duration_and_sr(file_path: Path):
    """
    Legge velocemente i metadati audio da file locale:
    - durata in secondi
    - sample rate
    """
    info = sf.info(str(file_path))
    duration_sec = info.frames / info.samplerate
    return duration_sec, info.samplerate


def scan_folder_dataset(source_name: str, source_dir: Path):
    rows = []

    if not source_dir.exists():
        print(f"[WARNING] Folder not found: {source_dir}")
        return rows

    # Ogni sottocartella rappresenta una classe
    class_dirs = [p for p in source_dir.iterdir() if p.is_dir()]

    # Barra sulle classi
    for class_dir in tqdm(class_dirs, desc=f"{source_name} classes"):
        mapped_label = class_dir.name
        original_label = class_dir.name

        keep_label = mapped_label in VALID_LABELS
        exclude_reason_label = "" if keep_label else "invalid_folder_label"

        # Lista file della classe corrente
        file_list = list(class_dir.rglob(f"*{AUDIO_EXTENSION}"))

        # Barra sui file della classe corrente
        for file_path in tqdm(file_list, desc=f"{source_name}/{class_dir.name}", leave=False):
            keep = keep_label
            exclude_reason = exclude_reason_label

            try:
                duration_sec, sample_rate = get_audio_duration_and_sr(file_path)
                status = "pending"
            except Exception as e:
                duration_sec = None
                sample_rate = None
                keep = False
                exclude_reason = f"read_error: {str(e)}"
                status = "failed_read"

            rows.append({
                "source": source_name,
                "filepath": str(file_path),
                "filename": file_path.name,
                "original_label": original_label,
                "original_label_id": None,
                "mapped_label": mapped_label if keep_label else None,
                "duration_sec": duration_sec,
                "sample_rate": sample_rate,
                "keep": keep,
                "exclude_reason": exclude_reason,
                "split": None,
                "use_for_model": False,
                "status": status,
            })

    return rows


def scan_huggingface_dataset(hf_dir: Path):
    rows = []

    if not hf_dir.exists():
        print(f"[WARNING] HuggingFace path not found: {hf_dir}")
        return rows

    dataset = load_from_disk(str(hf_dir))

    for i in tqdm(range(len(dataset)), desc="huggingface samples"):
        sample = dataset[i]
        label_id = sample["label"]
        mapped_label = HF_LABEL_MAP.get(label_id, None)

        keep = mapped_label is not None
        exclude_reason = "" if keep else "hf_label_1_ambiguous"

        try:
            audio_obj = sample["audio"]
            audio_array = audio_obj["array"]
            sample_rate = audio_obj["sampling_rate"]
            duration_sec = len(audio_array) / sample_rate
            status = "pending"
        except Exception as e:
            duration_sec = None
            sample_rate = None
            keep = False
            exclude_reason = f"read_error: {str(e)}"
            status = "failed_read"

        filepath_value = sample["file"] if ("file" in sample and sample["file"] is not None) else f"hf_index_{i}"
        filename_value = Path(filepath_value).name if isinstance(filepath_value, str) else f"hf_index_{i}.wav"

        rows.append({
            "source": "huggingface",
            "filepath": str(filepath_value),
            "filename": filename_value,
            "original_label": f"hf_label_{label_id}",
            "original_label_id": label_id,
            "hf_row_index": i,
            "mapped_label": mapped_label,
            "duration_sec": duration_sec,
            "sample_rate": sample_rate,
            "keep": keep,
            "exclude_reason": exclude_reason,
            "split": None,
            "use_for_model": False,
            "status": status,
        })

    return rows

In [ ]:
# ============================================================
# AGGIORNA SOLO LA PARTE HUGGINGFACE DEI METADATI
# ============================================================

temp = True
if not temp:
  start_time = time.perf_counter()

  # 1. Carica il metadata già esistente
  metadata_df = pd.read_csv(METADATA_CSV)

  # 2. Tieni solo Kaggle + Melaudis
  metadata_without_hf = metadata_df[metadata_df["source"] != "huggingface"].copy()

  # 3. Rigenera solo le righe HuggingFace
  hf_rows = scan_huggingface_dataset(HUGGINGFACE_DIR)
  hf_df = pd.DataFrame(hf_rows)

  # 4. Unisci la parte vecchia (kaggle+melaudis) con la nuova HF
  metadata_df = pd.concat([metadata_without_hf, hf_df], axis=0)

  # 5. Ordina di nuovo
  metadata_df = metadata_df.sort_values(
      by=["source", "mapped_label", "filename"],
      na_position="last"
  ).reset_index(drop=True)

  # 6. Salva
  metadata_df.to_csv(METADATA_CSV, index=False)

  end_time = time.perf_counter()

  print("Saved metadata to:")
  print(METADATA_CSV)
  print()

  print(f"Metadata update completed in {end_time - start_time:.2f} seconds")
  print()

  print("Total rows:", len(metadata_df))
  print("Rows kept:", metadata_df["keep"].sum())
  print("Rows excluded:", (~metadata_df["keep"]).sum())
  print()

  print("Kept rows by source:")
  print(metadata_df[metadata_df["keep"]]["source"].value_counts())
  print()

  print("hf_row_index presente?", "hf_row_index" in metadata_df.columns)

In [ ]:
# ============================================================
# COSTRUZIONE DEI METADATI ORIGINALI
# ============================================================

CONFIG_DONE = True
if not CONFIG_DONE:

    start_time = time.perf_counter()

    all_rows = []

    all_rows.extend(scan_folder_dataset("kaggle", KAGGLE_DIR))
    all_rows.extend(scan_folder_dataset("melaudis", MELAUDIS_DIR))
    all_rows.extend(scan_huggingface_dataset(HUGGINGFACE_DIR))

    metadata_df = pd.DataFrame(all_rows)

    metadata_df = metadata_df.sort_values(
        by=["source", "mapped_label", "filename"],
        na_position="last"
    ).reset_index(drop=True)

    metadata_df.to_csv(METADATA_CSV, index=False)

    end_time = time.perf_counter()

    print("Saved metadata to:")
    print(METADATA_CSV)
    print()

    print(f"Metadata build completed in {end_time - start_time:.2f} seconds")
    print()

    print("Total rows:", len(metadata_df))
    print("Rows kept:", metadata_df["keep"].sum())
    print("Rows excluded:", (~metadata_df["keep"]).sum())
    print()

    print("Kept rows by source:")
    print(metadata_df[metadata_df["keep"]]["source"].value_counts())
    print()

    print("Kept rows by mapped_label:")
    print(metadata_df[metadata_df["keep"]]["mapped_label"].value_counts())
    print()

    print("Excluded rows by reason:")
    print(metadata_df.loc[~metadata_df["keep"], "exclude_reason"].value_counts(dropna=False))

    metadata_df.head(10)


In [ ]:
# ============================================================
# CONFIGURAZIONE SPLIT E UNDERSAMPLING
# ============================================================

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_STATE = 42

# Undersampling moderato:
# se una classe nel train supera questo numero, la riduciamo
MAX_TRAIN_SAMPLES_PER_CLASS = 2000

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9, "Ratios must sum to 1."

In [ ]:
# ============================================================
# LETTURA METADATA E SPLIT STRATIFICATO
# ============================================================

df = pd.read_csv(METADATA_CSV)


# Reset colonne split/use_for_model per sicurezza
df["split"] = None
df["use_for_model"] = False

# Consideriamo solo i file validi
df_valid = df[df["keep"] == True].copy()

print("Total valid rows:", len(df_valid))
print()
print("Valid rows by class:")
print(df_valid["mapped_label"].value_counts())
print()

# Primo step: train vs temp(val+test)
# Manteniamo la stessa distribuzione delle classi
train_df, temp_df = train_test_split(
    df_valid,
    test_size=(VAL_RATIO + TEST_RATIO),
    stratify=df_valid["mapped_label"],
    random_state=RANDOM_STATE
)

# Secondo step: temp in val e test
val_relative_ratio = VAL_RATIO / (VAL_RATIO + TEST_RATIO)

val_df, test_df = train_test_split(
    temp_df,
    test_size=(1 - val_relative_ratio),
    stratify=temp_df["mapped_label"],
    random_state=RANDOM_STATE
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

In [ ]:
# ============================================================
# UNDERSAMPLING MODERATO SOLO SUL TRAINING
# ============================================================

print("Train counts before undersampling:")
print(train_df["mapped_label"].value_counts())
print()

sampled_parts = []

# Per ogni classe nel train:
# - se ha meno di MAX_TRAIN_SAMPLES_PER_CLASS esempi, la teniamo tutta
# - se ne ha di più, campioniamo casualmente fino al cap
for label, group in train_df.groupby("mapped_label"):
    n_keep = min(len(group), MAX_TRAIN_SAMPLES_PER_CLASS)
    sampled_parts.append(group.sample(n=n_keep, random_state=RANDOM_STATE))

train_under_df = pd.concat(sampled_parts, axis=0).copy()

# Questi sample saranno davvero usati dal modello
train_under_df["use_for_model"] = True
val_df["use_for_model"] = True
test_df["use_for_model"] = True

# Indici dei sample train non selezionati dopo undersampling
train_not_selected_idx = train_df.index.difference(train_under_df.index)

In [ ]:
# ============================================================
# SCRITTURA SPLIT FINALE NEL DATAFRAME COMPLETO
# ============================================================

# Assegniamo gli split
df.loc[train_df.index, "split"] = "train"
df.loc[val_df.index, "split"] = "val"
df.loc[test_df.index, "split"] = "test"

# Assegniamo i sample effettivamente usati dal modello
df.loc[train_under_df.index, "use_for_model"] = True
df.loc[val_df.index, "use_for_model"] = True
df.loc[test_df.index, "use_for_model"] = True

# I sample train non selezionati restano nel CSV,
# ma con use_for_model = False
df.loc[train_not_selected_idx, "use_for_model"] = False

# Salviamo il CSV finale
df.to_csv(METADATA_SPLIT_CSV, index=False)


In [ ]:
# ============================================================
# CONTROLLI SULLO SPLIT FINALE
# ============================================================

print("All valid rows by split:")
print(df[df["keep"] == True]["split"].value_counts())
print()

print("All valid rows class distribution per split:")
print(pd.crosstab(df[df["keep"] == True]["split"], df[df["keep"] == True]["mapped_label"]))
print()

print("Rows actually used for model:")
print(df[(df["keep"] == True) & (df["use_for_model"] == True)]["split"].value_counts())
print()

print("Class distribution of TRAIN after undersampling:")
print(
    df[
        (df["keep"] == True) &
        (df["use_for_model"] == True) &
        (df["split"] == "train")
    ]["mapped_label"].value_counts()
)
print()

print("Class distribution of VAL (unchanged):")
print(
    df[
        (df["keep"] == True) &
        (df["use_for_model"] == True) &
        (df["split"] == "val")
    ]["mapped_label"].value_counts()
)
print()

print("Class distribution of TEST (unchanged):")
print(
    df[
        (df["keep"] == True) &
        (df["use_for_model"] == True) &
        (df["split"] == "test")
    ]["mapped_label"].value_counts()
)
print()

print("Excluded rows still have split =")
print(df[df["keep"] == False]["split"].unique())




In [ ]:
# ============================================================
# CONFIGURAZIONE AUDIO E LETTURA CSV FINALE
# ============================================================

# Parametri audio comuni
#TARGET_SR = 16000
#N_MELS = 128
#N_FFT = 1024
#HOP_LENGTH = 512


#Secondo test con parametri audio diversi
#TARGET_SR = 22050
#N_MELS = 64
#N_FFT = 512
#HOP_LENGTH = 256

#Terzo test con parametri audio diversi
TARGET_SR = 22050
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 256


# Parametri DataLoader
BATCH_SIZE = 16
NUM_WORKERS = 0   # su Mac/Jupyter meglio 0

# Leggiamo il CSV finale
df = pd.read_csv(METADATA_SPLIT_CSV)

# Teniamo solo i file:
# - validi
# - realmente usati dal modello
df_model = df[(df["keep"] == True) & (df["use_for_model"] == True)].copy()

# Separiamo train / val / test
train_df = df_model[df_model["split"] == "train"].reset_index(drop=True)
val_df   = df_model[df_model["split"] == "val"].reset_index(drop=True)
test_df  = df_model[df_model["split"] == "test"].reset_index(drop=True)

print("Train size:", len(train_df))
print("Val size  :", len(val_df))
print("Test size :", len(test_df))
print()

print("Train distribution:")
print(train_df["mapped_label"].value_counts())
print()

print("Val distribution:")
print(val_df["mapped_label"].value_counts())
print()

print("Test distribution:")
print(test_df["mapped_label"].value_counts())

In [ ]:
# ============================================================
# LABEL MAPPING
# ============================================================

label_to_idx = {
    "Car": 0,
    "Bus": 1,
    "MC": 2,
    "Tram": 3,
    "Truck": 4
}

idx_to_label = {v: k for k, v in label_to_idx.items()}

print(label_to_idx)

In [ ]:
# ============================================================
# CARICAMENTO DATASET HUGGINGFACE
# ============================================================

hf_dataset = load_from_disk(str(HUGGINGFACE_DIR))

print(hf_dataset)
print("Numero sample HF:", len(hf_dataset))

In [ ]:
# ============================================================
# DATASET PYTORCH
# ============================================================

class AudioDataset(Dataset):
    def __init__(
        self,
        dataframe,
        label_to_idx,
        target_sr=16000,
        n_mels=128,
        n_fft=1024,
        hop_length=512,
        hf_dataset=None
    ):
        # Salviamo una copia del dataframe
        self.df = dataframe.reset_index(drop=True)

        # Mapping classe testuale -> indice numerico
        self.label_to_idx = label_to_idx

        # Sample rate target comune
        self.target_sr = target_sr

        # Dataset HuggingFace caricato fuori dalla classe
        self.hf_dataset = hf_dataset

        # Trasformazione in mel spectrogram
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=target_sr,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels
        )

        # Conversione in scala logaritmica (dB)
        self.db_transform = torchaudio.transforms.AmplitudeToDB()

        # Cache dei resampler per non ricrearli ogni volta
        self.resamplers = {}

    def __len__(self):
        return len(self.df)

    def _get_resampler(self, orig_sr):
        if orig_sr not in self.resamplers:
            self.resamplers[orig_sr] = torchaudio.transforms.Resample(
                orig_freq=orig_sr,
                new_freq=self.target_sr
            )
        return self.resamplers[orig_sr]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        filepath = row["filepath"]
        label_name = row["mapped_label"]
        source = row["source"]

        label_idx = self.label_to_idx[label_name]

        # --------------------------------------------------------
        # 1. LOAD AUDIO
        # --------------------------------------------------------
        if source == "huggingface":
            # Per HuggingFace non usiamo sf.read(filepath),
            # ma recuperiamo il sample direttamente dal Dataset HF
            if self.hf_dataset is None:
                raise ValueError("hf_dataset is required for HuggingFace samples.")

            hf_idx = int(row["hf_row_index"])
            sample = self.hf_dataset[hf_idx]

            audio_obj = sample["audio"]
            waveform = audio_obj["array"]
            sr = audio_obj["sampling_rate"]

            # Se il waveform è multi-canale, facciamo media sui canali
            if waveform.ndim == 2:
                waveform = waveform.mean(axis=1)

            waveform = torch.tensor(waveform, dtype=torch.float32)

        else:
            # Kaggle e Melaudis: filepath reale su disco
            waveform, sr = sf.read(filepath)

            if waveform.ndim == 2:
                waveform = waveform.mean(axis=1)

            waveform = torch.tensor(waveform, dtype=torch.float32)

        # --------------------------------------------------------
        # 2. RESAMPLE
        # --------------------------------------------------------
        # Portiamo tutti i file allo stesso sample rate
        if sr != self.target_sr:
            resampler = self._get_resampler(sr)
            waveform = resampler(waveform)

        # Da [time] a [1, time]
        waveform = waveform.unsqueeze(0)

        # --------------------------------------------------------
        # 3. LOG-MEL SPECTROGRAM
        # --------------------------------------------------------
        mel = self.mel_transform(waveform)
        mel_db = self.db_transform(mel)

        # --------------------------------------------------------
        # 4. NORMALIZZAZIONE PER SAMPLE
        # --------------------------------------------------------
        mean = mel_db.mean()
        std = mel_db.std()
        mel_db = (mel_db - mean) / (std + 1e-8)

        # --------------------------------------------------------
        # 5. RETURN
        # --------------------------------------------------------
        return {
            "x": mel_db,   # shape: [1, n_mels, time]
            "y": torch.tensor(label_idx, dtype=torch.long),
            "filepath": filepath,
            "label_name": label_name,
            "source": source
        }

In [ ]:
# ============================================================
# COLLATE FUNCTION CON PADDING
# ============================================================

def pad_collate_fn(batch):
    # Lista degli input
    xs = [item["x"] for item in batch]   # ciascuno: [1, n_mels, time]

    # Stack delle label
    ys = torch.stack([item["y"] for item in batch])

    # Troviamo la lunghezza temporale massima nel batch
    max_time = max(x.shape[-1] for x in xs)

    padded_xs = []
    lengths = []

    for x in xs:
        time_len = x.shape[-1]
        lengths.append(time_len)

        # Padding a destra sulla dimensione temporale
        pad_amount = max_time - time_len
        if pad_amount > 0:
            x = torch.nn.functional.pad(x, (0, pad_amount))

        padded_xs.append(x)

    # Stack finale del batch
    x_batch = torch.stack(padded_xs)  # [B, 1, n_mels, max_time]
    lengths = torch.tensor(lengths, dtype=torch.long)

    return {
        "x": x_batch,
        "y": ys,
        "lengths": lengths,
        "filepaths": [item["filepath"] for item in batch],
        "label_names": [item["label_name"] for item in batch]
    }

In [ ]:
# ============================================================
# CREAZIONE DEI DATASET
# ============================================================

train_dataset = AudioDataset(
    train_df,
    label_to_idx,
    target_sr=TARGET_SR,
    n_mels=N_MELS,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    hf_dataset=hf_dataset
)

val_dataset = AudioDataset(
    val_df,
    label_to_idx,
    target_sr=TARGET_SR,
    n_mels=N_MELS,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    hf_dataset=hf_dataset
)

test_dataset = AudioDataset(
    test_df,
    label_to_idx,
    target_sr=TARGET_SR,
    n_mels=N_MELS,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    hf_dataset=hf_dataset
)

print("Train dataset:", len(train_dataset))
print("Val dataset  :", len(val_dataset))
print("Test dataset :", len(test_dataset))

In [ ]:
# ============================================================
# CREAZIONE DEI DATALOADER
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=pad_collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=pad_collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=pad_collate_fn
)

print("DataLoaders creati correttamente.")

In [ ]:
# ============================================================
# QUICK BATCH TEST
# ============================================================

batch = next(iter(train_loader))

print("x shape      :", batch["x"].shape)       # [B, 1, n_mels, max_time]
print("y shape      :", batch["y"].shape)       # [B]
print("lengths shape:", batch["lengths"].shape) # [B]

print("Prime 10 label numeriche:")
print(batch["y"][:10])

print("\nPrime 2 path:")
print(batch["filepaths"][:2])

In [ ]:
# ============================================================
# SPATIAL PYRAMID POOLING
# ============================================================

class SpatialPyramidPooling(nn.Module):
    def __init__(self, levels=(1, 2, 4), pool_type="max"):
        super().__init__()
        self.levels = levels
        self.pool_type = pool_type

    def forward(self, x):
        # x ha shape [B, C, H, W]
        pooled_outputs = []

        for level in self.levels:
            if self.pool_type == "max":
                pooled = F.adaptive_max_pool2d(x, output_size=(level, level))
            elif self.pool_type == "avg":
                pooled = F.adaptive_avg_pool2d(x, output_size=(level, level))
            else:
                raise ValueError("pool_type must be 'max' or 'avg'")

            # Da [B, C, level, level] a [B, C * level * level]
            pooled = pooled.view(x.size(0), -1)
            pooled_outputs.append(pooled)

        # Concateniamo tutti i livelli
        out = torch.cat(pooled_outputs, dim=1)
        return out

In [ ]:
# ============================================================
# MODELLO CNN + SPP
# ============================================================

class CNN_SPP_Classifier(nn.Module):
    def __init__(self, num_classes=5, spp_levels=(1, 2, 4), dropout=0.3):
        super().__init__()

        # --------------------------------------------------------
        # BACKBONE CNN
        # --------------------------------------------------------
        # Input: [B, 1, 128, T]
        self.features = nn.Sequential(
            # Blocco 1
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            # Blocco 2
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            # Blocco 3
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        # Layer SPP
        self.spp = SpatialPyramidPooling(levels=spp_levels, pool_type="max")

        # Calcolo della dimensione di output dello SPP
        spp_output_dim = 64 * sum(level * level for level in spp_levels)

        # Classificatore finale
        self.classifier = nn.Sequential(
            nn.Linear(spp_output_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # x: [B, 1, 128, T]
        x = self.features(x)      # [B, 64, H', W']
        x = self.spp(x)           # [B, fixed_dim]
        x = self.classifier(x)    # [B, num_classes]
        return x

In [ ]:
# ============================================================
# QUICK MODEL TEST
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = CNN_SPP_Classifier(
    num_classes=5,
    spp_levels=(1, 2, 4),
    dropout=0.3
).to(device)

# Prendiamo un batch dal train_loader
batch = next(iter(train_loader))

x = batch["x"].to(device)   # [B, 1, 128, T]
y = batch["y"].to(device)   # [B]

print("Input batch shape :", x.shape)
print("Target shape      :", y.shape)

with torch.no_grad():
    logits = model(x)

print("Output logits shape:", logits.shape)   # [B, 5]

In [ ]:
# ============================================================
# LOSS, OPTIMIZER E CONFIG TRAINING
# ============================================================

# File dove salveremo il miglior modello trovato su validation
BEST_MODEL_PATH = PROJECT_DIR / "best_model_cnn_spp.pth"

# Numero di epoche iniziale
#NUM_EPOCHS = 10
NUM_EPOCHS = 6


# Learning rate iniziale
LEARNING_RATE = 1e-3

# Loss per classificazione multiclasse
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("Best model path :", BEST_MODEL_PATH)
print("Num epochs      :", NUM_EPOCHS)
print("Learning rate   :", LEARNING_RATE)

In [ ]:

# ============================================================
# TRAINING DI UNA EPOCA
# ============================================================

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """
    Esegue una singola epoca di training.

    Restituisce:
    - loss media sull'epoca
    - accuracy media sull'epoca
    """
    model.train()  # modalità training

    running_loss = 0.0
    correct = 0
    total = 0

    # Barra di avanzamento sui batch
    progress_bar = tqdm(dataloader, desc="Training", leave=False)

    for batch in progress_bar:
        x = batch["x"].to(device)   # [B, 1, n_mels, T]
        y = batch["y"].to(device)   # [B]

        # Azzera i gradienti prima del nuovo step
        optimizer.zero_grad()

        # Forward
        logits = model(x)

        # Loss
        loss = criterion(logits, y)

        # Backpropagation
        loss.backward()

        # Aggiorna i pesi
        optimizer.step()

        # Accumulo metriche
        batch_size = y.size(0)
        running_loss += loss.item() * batch_size

        preds = torch.argmax(logits, dim=1)
        correct += (preds == y).sum().item()
        total += batch_size

        # Metriche correnti mostrate nella barra
        current_loss = running_loss / total
        current_acc = correct / total

        progress_bar.set_postfix({
            "loss": f"{current_loss:.4f}",
            "acc": f"{current_acc:.4f}"
        })

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [ ]:
def evaluate(model, dataloader, criterion, device, desc="Evaluation"):

    """

    Valuta il modello senza aggiornare i pesi.

    Restituisce:

    - loss media

    - accuracy media

    """

    model.eval()

    running_loss = 0.0

    correct = 0

    total = 0

    progress_bar = tqdm(dataloader, desc=desc, leave=False)

    with torch.no_grad():

        for batch in progress_bar:

            x = batch["x"].to(device)

            y = batch["y"].to(device)

            logits = model(x)

            loss = criterion(logits, y)

            batch_size = y.size(0)

            running_loss += loss.item() * batch_size

            preds = torch.argmax(logits, dim=1)

            correct += (preds == y).sum().item()

            total += batch_size

            current_loss = running_loss / total

            current_acc = correct / total

            progress_bar.set_postfix({

                "loss": f"{current_loss:.4f}",

                "acc": f"{current_acc:.4f}"

            })

    epoch_loss = running_loss / total

    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [ ]:


temp = True
if temp:
    PROJECT_DIR = Path("/content/drive/MyDrive/AudioFingerprinting")
    CSV_PATH = PROJECT_DIR / "Metadata/metadata_original_files_split.csv"

    df = pd.read_csv(CSV_PATH)

    df_model = df[(df["keep"] == True) & (df["use_for_model"] == True)].copy()

    bad_indices = []
    bad_files = []

    for idx, row in df_model.iterrows():
        source = row["source"]

        # Per HuggingFace non usiamo sf.read(filepath)
        if source == "huggingface":
            continue

        filepath = row["filepath"]

        try:
            sf.info(filepath)
        except Exception as e:
            bad_indices.append(idx)
            bad_files.append((filepath, str(e)))

    print("Numero file locali non leggibili:", len(bad_files))
    print()

    for x in bad_files[:20]:
        print(x)

In [ ]:
# ============================================================
# TRAINING LOOP COMPLETO
# ============================================================

import time

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_acc = 0.0
best_epoch = -1

for epoch in range(NUM_EPOCHS):
    start_epoch = time.perf_counter()

    # -------------------------
    # TRAIN
    # -------------------------
    train_loss, train_acc = train_one_epoch(
        model=model,
        dataloader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    # -------------------------
    # VALIDATION
    # -------------------------
    val_loss, val_acc = evaluate(
    model=model,
    dataloader=val_loader,
    criterion=criterion,
    device=device,
    desc="Validation"
    )

    # -------------------------
    # SALVA METRICHE
    # -------------------------
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    # -------------------------
    # BEST MODEL CHECK
    # -------------------------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1

        # Salva solo i pesi del modello
        torch.save(model.state_dict(), BEST_MODEL_PATH)

    end_epoch = time.perf_counter()

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print(f"  Best Val Acc so far: {best_val_acc:.4f} (epoch {best_epoch})")
    print(f"  Epoch time: {end_epoch - start_epoch:.2f} sec")
    print("-" * 50)

print("Training completed.")
print(f"Best model saved at: {BEST_MODEL_PATH}")
print(f"Best validation accuracy: {best_val_acc:.4f}")
print(f"Best epoch: {best_epoch}")

In [ ]:
# ============================================================
# STAMPA HISTORY
# ============================================================

history_df = pd.DataFrame(history)
history_df.index = history_df.index + 1
history_df.index.name = "epoch"

history_df

In [ ]:
# ============================================================
# RICARICA DEL MIGLIOR MODELLO
# ============================================================

best_model = CNN_SPP_Classifier(
    num_classes=5,
    spp_levels=(1, 2, 4),
    dropout=0.3
).to(device)

best_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
best_model.eval()

print("Best model loaded correctly.")

In [ ]:
# ============================================================
# TEST FINALE
# ============================================================

test_loss, test_acc = evaluate(
    model=best_model,
    dataloader=test_loader,
    criterion=criterion,
    device=device
)

print("Final test results:")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test Acc : {test_acc:.4f}")

In [ ]:
# ============================================================
# RACCOLTA PREDIZIONI SU TEST SET
# ============================================================

best_model.eval()

all_true = []
all_pred = []

with torch.no_grad():
    for batch in test_loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)

        logits = best_model(x)
        preds = torch.argmax(logits, dim=1)

        all_true.extend(y.cpu().numpy())
        all_pred.extend(preds.cpu().numpy())

all_true = np.array(all_true)
all_pred = np.array(all_pred)

print("Numero totale sample test:", len(all_true))
print("Prime 10 label vere     :", all_true[:10])
print("Prime 10 label predette :", all_pred[:10])

In [ ]:
# ============================================================
# CONFUSION MATRIX NUMERICA
# ============================================================

label_order = ["Car", "Bus", "MC", "Tram", "Truck"]
label_indices = [label_to_idx[x] for x in label_order]

cm = confusion_matrix(all_true, all_pred, labels=label_indices)

cm_df = pd.DataFrame(cm, index=label_order, columns=label_order)

print("Confusion Matrix:")
print(cm_df)

In [ ]:
# ============================================================
# PLOT CONFUSION MATRIX
# ============================================================

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix - Test Set")
plt.colorbar()

tick_marks = np.arange(len(label_order))
plt.xticks(tick_marks, label_order, rotation=45)
plt.yticks(tick_marks, label_order)

# scrive i numeri dentro le celle
threshold = cm.max() / 2.0 if cm.max() > 0 else 0

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j, i, format(cm[i, j], "d"),
            ha="center", va="center",
            color="white" if cm[i, j] > threshold else "black"
        )

plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

report_text = classification_report(
    all_true,
    all_pred,
    labels=label_indices,
    target_names=label_order,
    digits=4
)

print(report_text)

In [ ]:
# ============================================================
# CLASSIFICATION REPORT IN DATAFRAME
# ============================================================

report_dict = classification_report(
    all_true,
    all_pred,
    labels=label_indices,
    target_names=label_order,
    output_dict=True,
    digits=4
)

report_df = pd.DataFrame(report_dict).transpose()
report_df